---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 4.3 — Conversation Simulation

---

## 💬 Product Check-in

> **Sam (Product):** "Our eval numbers look great, but I'm getting support tickets that tell a different story. Users say the agent forgets what they asked two messages ago. One person asked about `mlflow.genai.evaluate`, then asked a follow-up about its scorers parameter, and the agent acted like it had never heard of the function."
>
> **Sam:** "Can we just write multi-turn test cases by hand?"
>
---

**Objective:** Use MLflow's `ConversationSimulator` to generate synthetic multi-turn conversations, then evaluate them with session-level scorers that catch problems invisible to single-turn eval.

In this notebook:
1. Define simulation test cases with goals and personas
2. Run `ConversationSimulator` to generate multi-turn conversations
3. Score sessions with `ConversationCompleteness`, `UserFrustration`, and `KnowledgeRetention`
4. Compare what session-level eval catches vs. single-turn eval

---
## 0 — Connect to MLflow

In [ ]:
from functools import cached_property
from pathlib import Path
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ENV_FILE = Path.cwd().parent.parent / ".env"  # adjust .parent count to match your nesting

class AgentConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",  # don't choke on unrelated vars in .env
    )

    # Secrets / endpoints (from .env)
    gemini_api_key: SecretStr
    gemini_openai_base_url: str

    # MLflow (from .env, with fallbacks)
    mlflow_tracking_uri: str
    experiment_name: str = Field(default="mlflow-agent-1", alias="EXPERIMENT_1_NAME")

    # Model knobs (hardcoded defaults, overridable via env if you want)
    llm: str = "gemini-2.5-flash-lite"
    temperature: float = 0.2
    max_tokens: int = 512
    ## Adding judge model to config
    judge_llm: str = "gemini:/gemini-3.1-flash-lite-preview"

    # Prompt
    prompt_name: str = "mlflow-agent-system"
    system_prompt: str = "You are an MLflow assistant."

    ##DATASET
    eval_dataset_name: str = "mlflow-agent-eval"

    ##ALIGN JUDGE MODEL TO CONFIG
    #Both the judge's assessments and human feedback must share the same assessment name 
    # on the same traces for alignment to work.
    align_judge: str = "response_quality"

    @cached_property
    def prompt_alias(self) -> str:
        return f"prompts:/{self.prompt_name}@prod"

cfg = AgentConfig()

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()
mlflow.set_experiment(cfg.experiment_name)
mlflow.langchain.autolog()

---
## 1 — The Agent

We'll use the same agent from our previous experiments — LangChain with tools. The key difference in this notebook is *how* we call it: the `ConversationSimulator` expects a `predict_fn` that accepts a `messages` list (chat completions format), not a single question string.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import mlflow
import requests
import importlib
import inspect

llm = ChatGoogleGenerativeAI(model=cfg.llm, 
                             temperature=cfg.temperature, 
                             max_tokens=cfg.max_tokens, 
                             #api_key=cfg.gemini_api_key.get_secret_value()
                             )


# ── Tools from previous experiments ────────────────────────────────────
def get_mlflow_version_pypi() -> str:
    """
    Fetches the current stable release of MLflow directly from PyPI.
    No API key required.
    """
    try:
        response = requests.get("https://pypi.org/pypi/mlflow/json", timeout=5)
        response.raise_for_status()
        return response.json()["info"]["version"]
    except Exception as e:
        return f"Could not fetch version from PyPI: {str(e)}"


def check_api_exists(fully_qualified_name: str) -> dict:
    """
    Check whether an MLflow API exists and return its signature.

    Args:
        fully_qualified_name: Dotted path to the API, e.g. "mlflow.set_experiment"

    Returns:
        A dict with 'exists', and if True, 'parameters' and 'description'.
    """
    parts = fully_qualified_name.rsplit(".", 1)
    if len(parts) != 2:
        return {"exists": False, "error": f"Invalid API path: {fully_qualified_name}"}
    module_path, attr_name = parts
    try:
        mod = importlib.import_module(module_path)
    except ModuleNotFoundError:
        return {"exists": False}
    obj = getattr(mod, attr_name, None)
    if obj is None:
        return {"exists": False}
    try:
        sig = inspect.signature(obj)
        parameters = list(sig.parameters.keys())
    except (ValueError, TypeError):
        parameters = []
    doc = inspect.getdoc(obj) or ""
    first_line = doc.split("\n")[0] if doc else "No description available."
    return {"exists": True, "name": fully_qualified_name, "parameters": parameters, "description": first_line}


SYSTEM_PROMPT = """You are a helpful MLflow assistant. Answer questions about MLflow features,
APIs, and best practices. Be concise and accurate. If you're unsure, say so.

You have access to tools:
- get_mlflow_version_pypi: Fetches the current MLflow release version from PyPI.
  Use this whenever the user asks about the current or latest MLflow version.
- check_api_exists: Checks whether a specific MLflow API exists and returns its signature.
  Use this when the user asks whether a function/class exists or asks about its parameters.
  Always use this tool instead of guessing API signatures.
"""

tools = [get_mlflow_version_pypi, check_api_exists]

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

In [ ]:
results = agent.invoke({"messages": {"role": "user", "content": "What is MLflow?"}})

### The `predict_fn` contract for conversation simulation

The `ConversationSimulator` calls your function once per turn, passing the full conversation history as a `messages` list. It also passes a `mlflow_session_id` kwarg to tie all turns in a conversation together.

In [ ]:
def multi_turn_predict_fn(messages: list[dict], **kwargs) -> str:
    """Multi-turn predict function for ConversationSimulator."""
    results = agent.invoke({"messages": messages})
    return results["messages"][-1].content

---
## 2 — The Gap: What Single-Turn Eval Misses

Let's run a quick single-turn eval to show the baseline. These pass — but they can't tell us anything about multi-turn behavior.

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery

single_turn_data = [
    {
        "inputs": {"question": "What is the current version of MLflow?"},
        "expectations": {
            "expected_facts": ["3"],
        },
    },
    {
        "inputs": {"question": "Does mlflow.genai.evaluate exist?"},
        "expectations": {
            "expected_facts": ["yes", "data", "scorers"],
        },
    },
]

def single_turn_predict(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

single_turn_results = mlflow.genai.evaluate(
    data=single_turn_data,
    predict_fn=single_turn_predict,
    scorers=[Correctness(model=cfg.judge_llm), RelevanceToQuery(model=cfg.judge_llm)],
)

print("=== Single-turn eval (baseline) ===")
print(single_turn_results.metrics)

> **Watch for it:** Single-turn scores look good. But none of these tests check whether the agent can maintain context across turns, complete a multi-step goal, or handle a frustrated user. That's what we'll test next.

---
## 3 — Defining Simulation Test Cases

Each test case describes a *goal* a simulated user is trying to achieve, plus an optional *persona* that shapes how they interact. The simulator generates realistic multi-turn conversations by having an LLM role-play as the user.

| Field | Purpose | Required |
|---|---|---|
| `goal` | What the simulated user is trying to accomplish |
| `persona` | How the user behaves (beginner, impatient, etc.) |
| `simulation_guidelines` | Constraints on how the simulated user should interact |
| `expectations` | Ground truth for session-level scorers |

In [ ]:
test_cases = [
    # ── Multi-step learning goal ───────────────────────────────────────
    {
        "goal": "Learn how to set up MLflow tracing for a LangChain application",
        "persona": "A data scientist who has used MLflow for experiment tracking but is new to tracing",
        "simulation_guidelines": [
            "Ask about tracing first, then ask a follow-up about how to view traces in the UI",
        ],
    },
    # ── Follow-up that requires remembering context ────────────────────
    {
        "goal": "Understand what parameters mlflow.genai.evaluate takes and then ask how to use it with custom scorers",
        "persona": "An ML engineer building an evaluation pipeline",
        "simulation_guidelines": [
            "First ask about the function's parameters",
            "Then reference the previous answer and ask about custom scorers",
        ],
    },
    # ── Troubleshooting flow ───────────────────────────────────────────
    {
        "goal": "Figure out why mlflow.start_run() is throwing an error about nested runs",
        "persona": "A frustrated developer who has been debugging for an hour",
        "simulation_guidelines": [
            "Start with a vague complaint about the error",
            "Only provide the full error message when asked",
        ],
    },
    # ── Version-specific question chain ────────────────────────────────
    {
        "goal": "Find out the current MLflow version and whether it supports the new genai module",
        "persona": "A beginner who just heard about MLflow at a conference",
        "simulation_guidelines": [
            "Ask about the version first, then ask what's new in MLflow 3",
        ],
    },
    # ── Out-of-scope deflection ────────────────────────────────────────
    {
        "goal": "Ask the MLflow assistant to help with a general Python debugging question unrelated to MLflow",
        "persona": "A developer who thinks the assistant is a general-purpose coding helper",
        "simulation_guidelines": [
            "Start by asking about a Python TypeError",
            "If redirected, push back once before accepting",
        ],
    },
]

print(f"Simulation test cases: {len(test_cases)}")

---
## 4 — Running the Conversation Simulator

The `ConversationSimulator` generates multi-turn conversations by pairing your agent with a simulated user. For each test case:

1. The simulated user sends a message working toward the `goal`
2. Your agent responds
3. An LLM checks whether the goal has been achieved — if so, the conversation ends early
4. Otherwise, the loop continues up to `max_turns`

Each turn is traced and tagged with `mlflow.trace.session` so all turns in a conversation are grouped together.

In [ ]:
from mlflow.genai.simulators import ConversationSimulator

simulator = ConversationSimulator(
    user_model=cfg.judge_llm,  # using judge as user to test alignment
    test_cases=test_cases,
    max_turns=5,
)

print(f"Simulator configured for {len(test_cases)} test cases, max {5} turns each")

### What the traces look like

Open the MLflow UI after running the simulation. Each conversation appears as a **session** — a group of traces sharing the same `mlflow.trace.session` ID:

```
SESSION: sim-abc123
│
├─ TRACE: Turn 1
│  └─ SPAN: Agent Executor (AGENT)
│     ├─ SPAN: ChatGoogleGenerativeAI (LLM)
│     └─ SPAN: ChatGoogleGenerativeAI (LLM)
│
├─ TRACE: Turn 2
│  └─ SPAN: Agent Executor (AGENT)
│     ├─ SPAN: ChatGoogleGenerativeAI (LLM)
│     ├─ SPAN: check_api_exists (TOOL)
│     └─ SPAN: ChatGoogleGenerativeAI (LLM)
│
└─ TRACE: Turn 3
   └─ SPAN: Agent Executor (AGENT)
      ├─ SPAN: ChatGoogleGenerativeAI (LLM)
      └─ SPAN: ChatGoogleGenerativeAI (LLM)
```

Session-level scorers evaluate the *entire group*, not individual traces.

---
## 5 — Session-Level Scorers

These scorers operate on entire conversations, not individual turns. They catch problems that single-turn scorers can't see.

| Scorer | What it measures | Return values |
|---|---|---|
| `ConversationCompleteness()` | Were all user requests fully addressed? | `"yes"` / `"no"` |
| `UserFrustration()` | Did the user express frustration? Was it resolved? | `"none"` / `"resolved"` / `"unresolved"` |
| `KnowledgeRetention()` | Does the agent correctly use information from earlier turns? | `"yes"` / `"no"` |
| `ConversationalRoleAdherence()` | Does the agent stay in its defined role throughout? | `"yes"` / `"no"` |

In [ ]:
from mlflow.genai.scorers import (
    ConversationCompleteness,
    UserFrustration,
    KnowledgeRetention,
    ConversationalRoleAdherence,
)

results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=multi_turn_predict_fn,
    scorers=[
        # ── Session-level quality ─────────────────────────────────
        ConversationCompleteness(model=cfg.judge_llm),
        UserFrustration(model=cfg.judge_llm),
        KnowledgeRetention(model=cfg.judge_llm),
        ConversationalRoleAdherence(model=cfg.judge_llm),
    ],
)

print("=== Conversation simulation results ===")
print(results.metrics)

In [ ]:
results.tables["eval_results_table"]

---
## 6 — Interpreting the Results

Look at the per-session breakdown above and check:

1. **Conversation Completeness:** Did the agent actually help the user achieve their goal? A `"no"` here means the conversation ended without the user getting what they needed — even if each individual response scored well on correctness.

2. **User Frustration:** The "frustrated developer" test case is designed to trigger frustration. `"resolved"` means the agent handled it well. `"unresolved"` means the user stayed frustrated — a real support ticket waiting to happen.

3. **Knowledge Retention:** Did the agent remember context from earlier turns? A `"no"` on the `mlflow.genai.evaluate` follow-up test case means the agent forgot the function it just looked up — exactly the bug Sam reported.

4. **Role Adherence:** The out-of-scope test case checks whether the agent stays in its MLflow assistant role when asked general Python questions.

### What to do with failures

| Failure | Root cause | Fix |
|---|---|---|
| Completeness: `"no"` | Agent gave partial answers or went off-track | Improve system prompt to emphasize thoroughness |
| Frustration: `"unresolved"` | Agent didn't acknowledge user's frustration or repeated unhelpful responses | Add empathy/acknowledgment to system prompt |
| Knowledge Retention: `"no"` | Agent not using conversation history effectively | Check that `messages` list is being passed correctly to the model |
| Role Adherence: `"no"` | Agent answered out-of-scope questions instead of redirecting | Tighten role boundaries in system prompt |

---
## 7 — Adding Conversational Guidelines

Beyond the built-in scorers, you can define custom guidelines that are evaluated across the full conversation. This is useful for domain-specific rules.

In [ ]:
from mlflow.genai.scorers import ConversationalGuidelines

guidelines_scorer = ConversationalGuidelines(
    guidelines=[
        "The assistant must never fabricate MLflow API names or parameter names",
        "The assistant must use the check_api_exists tool before describing any API signature",
        "The assistant must acknowledge when it does not know something rather than guessing",
    ],
)

guidelines_results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=multi_turn_predict_fn,
    scorers=[guidelines_scorer],
)

print("=== Guidelines evaluation ===")
print(guidelines_results.metrics)

---
## 8 — Single-Turn vs. Multi-Turn: What Did We Find?

Compare the two evaluation approaches side by side. The single-turn eval likely passes cleanly. The session-level eval should surface failures the single-turn eval is blind to:

| Eval type | What it catches | What it misses |
|---|---|---|
| Single-turn | Factual accuracy, tool correctness, relevance | Context loss, frustration, incomplete goals |
| Multi-turn (session) | Goal completion, frustration, knowledge retention, role drift | N/A — it's a superset |

> **The takeaway:** Single-turn eval is necessary but not sufficient. An agent can ace every individual question and still fail as a conversational product.

---
## 9 — Bonus: Generating Test Cases from Production Sessions

Writing simulation test cases by hand is fine to start, but as your agent gets production traffic you can bootstrap test cases from real conversations. `generate_test_cases` distills goals, personas, and guidelines from existing sessions.

In [ ]:
# Uncomment once you have production sessions:

# from mlflow.genai.simulators import generate_test_cases
#
# sessions = mlflow.search_sessions(
#     locations=[experiment_id],
#     max_results=20,
# )
#
# generated_cases = generate_test_cases(sessions)
# print(f"Generated {len(generated_cases)} test cases from production")
# for case in generated_cases[:3]:
#     print(f"  Goal: {case['goal']}")
#     print(f"  Persona: {case.get('persona', 'N/A')}")
#     print()

---
## Summary:

| Concept | How we used it |
|---|---|
| Product conversation | Sam's feedback → multi-turn conversations are failing despite good single-turn scores |
| `ConversationSimulator` | Generated synthetic multi-turn conversations from goal-based test cases |
| Test cases with goals + personas | Defined what the simulated user is trying to achieve and how they behave |
| `ConversationCompleteness()` | Scored whether the agent fully addressed the user's goal |
| `UserFrustration()` | Detected and classified user frustration across the session |
| `KnowledgeRetention()` | Caught cases where the agent forgot context from earlier turns |
| `ConversationalRoleAdherence()` | Verified the agent stayed in its MLflow assistant role |
| `ConversationalGuidelines()` | Evaluated custom domain rules across the full conversation |
| `generate_test_cases()` | Bootstrap test cases from real production sessions |

### The pattern

```
Product feedback ("users say the agent forgets context")
  → Identify the gap (single-turn eval can't test conversations)
  → Define simulation test cases with goals and personas
  → Run ConversationSimulator to generate multi-turn sessions
  → Score with session-level scorers (Completeness, Frustration, Retention)
  → Identify failures invisible to single-turn eval
  → Fix root causes (prompt, context handling, role boundaries)
  → Re-run simulation to confirm improvement
```

---